# Trabajo Práctico 2 - Grupo 02

### Modelo Random Forest

Integrantes:

*   Bermudez, Agustin
*   Calderón, Tiago
*   Gonzalez Pautaso, Mateo
*   Moreyra, Santiago
*   Nieves, Maylen

El objetivo es entrenar un modelo de Random Forest sobre dos representaciones de texto (TF-IDF y BoW) con búsqueda de hiperparámetros, comparando su desempeño contra el baseline de Naive Bayes (~0.66 F1-macro en Kaggle).

Random Forest es un ensemble de árboles de decisión entrenados sobre subconjuntos aleatorios del data y de las features. Al promediar muchos árboles distintos reduce la varianza y generaliza mejor que un árbol solo.

## Importación e instalación de dependencias


In [ ]:
!pip install -q spacy
!python -m spacy download es_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 85.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV

In [ ]:
from common.preprocessing import clean_classical
from common.data_utils import get_split, make_tfidf, make_bow, SEED
from common.evaluation import evaluate
from common.io_utils import save_model, load_model, make_submission

np.random.seed(SEED)

## Carga de datos y preprocesamiento

In [ ]:
train = pd.read_csv("/content/train.csv")
test  = pd.read_csv("/content/test.csv")
X_train_raw, X_val_raw, y_train, y_val = get_split(train)

In [ ]:
print("Preprocesando")
X_train = np.array([clean_classical(t) for t in X_train_raw])
X_val   = np.array([clean_classical(t) for t in X_val_raw])
X_test  = np.array([clean_classical(t) for t in test['text'].values])
print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

Preprocesando
Train: 40,800 | Val: 10,200 | Test: 8,500


## Entrega N4: Bayes Naive + TF-IDF con RandomizedSearch

Búsqueda de hiperparámetros sobre RF + TF-IDF con 30 iteraciones aleatorias.
Se usa RandomizedSearchCV en lugar de GridSearchCV porque el espacio de
hiperparámetros de RF es grande - el grid completo serían 144 combinaciones
× 5 folds = 720 fits, lo cual agota la RAM de Colab.

Los hiperparámetros de `ngram_range`, `sublinear_tf` y `max_features` están
fijados en base a los resultados de experimentos anteriores: `(1, 2)`, `False`
y `sqrt` respectivamente. La búsqueda se concentra en:

- `tfidf__min_df`: mínima frecuencia de documento (1, 2 o 3)
- `rf__n_estimators`: cantidad de árboles del ensemble (100, 200 o 300)
- `rf__max_depth`: profundidad máxima de cada árbol (80, 100 o 120)
- `rf__min_samples_leaf`: mínimo de muestras en una hoja (1, 2 o 3)

In [ ]:
pipe = Pipeline([
    ("tfidf", make_tfidf(ngram_range=(1, 2), sublinear_tf=False)),
    ("rf",    RandomForestClassifier(
                  max_features="sqrt",
                  class_weight="balanced",
                  random_state=SEED,
                  n_jobs=2))])  # 2 en lugar de -1 para evitar crash de memoria

param_dist_tfidf = {
    "tfidf__min_df": [1, 2, 3],
    "rf__n_estimators": [100, 200, 300],
    "rf__max_depth": [80, 100, 120],
    "rf__min_samples_leaf": [1, 2, 3],
}

rs_rf_tfidf = RandomizedSearchCV(
    pipe,
    param_dist_tfidf,
    n_iter=30,
    cv=5,
    scoring="f1_macro",
    n_jobs=2,
    random_state=SEED,
    verbose=1
)

print("Buscando mejores hiperparámetros RF + TF-IDF")
rs_rf_tfidf.fit(X_train, y_train)

print(f"\nMejores hiperparámetros: {rs_rf_tfidf.best_params_}")
print(f"Mejor F1-macro en CV: {rs_rf_tfidf.best_score_:.4f}")

y_pred = rs_rf_tfidf.best_estimator_.predict(X_val)
evaluate("rf_tfidf_randomsearch_v4_2", y_val, y_pred, hyperparams=rs_rf_tfidf.best_params_)

best_pipe = rs_rf_tfidf.best_estimator_
best_pipe.fit(np.concatenate([X_train, X_val]), np.concatenate([y_train, y_val]))
save_model(best_pipe, "rf_tfidf_randomsearch_v4_2")
make_submission(best_pipe, pd.DataFrame({"id": test["id"], "text": X_test}), "submission_rf_tfidf_randomsearch_v4_2.csv");

Buscando mejores hiperparámetros RF + TF-IDF
Fitting 5 folds for each of 30 candidates, totalling 150 fits


KeyboardInterrupt: 